# Cell 1: Environment Setup

In [1]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v13 — PART 0: v12's Mamba configuration sweep is CONSOLIDATED into one fixed,
# permanent fusion backbone — 2 blocks x d_state 16 x d_conv 2 (expand 2, fused
# causal-conv1d path) — the sweep / conv-implementation scaffolding is removed.
# PART 1: ablation of the two auxiliary loss weights (lambda_align, lambda_cons).
# PART 2: a small grid ONLY around the weight(s) Part 1 shows are REAL.
# Everything else FIXED. SEED is the single source for all seeding here.
# ==============================================================================
SEED = 42
LACA_BETA = 1.0   # LACA temporal_bias_scale — FIXED (v9 sweep winner)
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)",
    "Audio time positions": "Ta = 64 per window (FIXED, v7)",
    "Sensor input":        "raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)",
    "LACA beta":           f"temporal_bias_scale = {LACA_BETA} (FIXED, v9 sweep winner)",
    "LACA g(Δτ)":          "mlp Linear(1,32)-Tanh-Linear(32,1), 97 params (FIXED; v10 form + capacity comparisons)",
    "LACA heads":          "num_heads = 4 (FIXED, conservative default kept after v11)",
    "Mamba backbone":      "2 blocks x d_state 16 x d_conv 2, expand 2, fused conv path — 874,496 params "
                           "(FIXED, v12 sweep winner)",
    "PART 1 ABLATION":     "loss weights: baseline (0.1, 0.1) | no_align (0.0, 0.1) | no_cons (0.1, 0.0)",
    "PART 2 GRID":         "{0.05, 0.1, 0.2} ONLY for the weight(s) Part 1 shows are REAL",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; no Hann)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seed":                f"{SEED} (random / numpy / torch / cuda, cuDNN deterministic; reset for every condition)",
    "Everything else":     "v12 unchanged (sensor path, split, 20 epochs, AdamW, eval)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v13 — CONSOLIDATED MAMBA + AUXILIARY LOSS-WEIGHT ABLATION")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction for every condition of the loss-weight ablation / grid (Cell 6).
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

  AEROSYNC-MAMBA v13 — CONSOLIDATED MAMBA + AUXILIARY LOSS-WEIGHT ABLATION
  • Audio bin selection : strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)
  • Audio time positions: Ta = 64 per window (FIXED, v7)
  • Sensor input        : raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)
  • LACA beta           : temporal_bias_scale = 1.0 (FIXED, v9 sweep winner)
  • LACA g(Δτ)          : mlp Linear(1,32)-Tanh-Linear(32,1), 97 params (FIXED; v10 form + capacity comparisons)
  • LACA heads          : num_heads = 4 (FIXED, conservative default kept after v11)
  • Mamba backbone      : 2 blocks x d_state 16 x d_conv 2, expand 2, fused conv path — 874,496 params (FIXED, v12 sweep winner)
  • PART 1 ABLATION     : loss weights: baseline (0.1, 0.1) | no_align (0.0, 0.1) | no_cons (0.1, 0.0)
  • PART 2 GRID         : {0.05, 0.1, 0.2} ONLY for the weight(s) Part 1 shows are REAL
  • Audio windowing     : none (raw 1536-sample

# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba) — mlp g(Δτ) 97 params, FIXED Mamba backbone 2 x d_state 16 x d_conv 2 (v12 winner)

In [2]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
#
# v13 — the fusion backbone is FIXED at the v12 sweep winner (Stage A num_blocks,
# Stage B d_state, Stage C d_conv): 2 blocks x d_state 16 x d_conv 2, expand 2.
# d_conv=2 is inside the causal-conv1d kernel's supported widths (2..4), so
# mamba_ssm takes its natural fused path — exactly as the winning v12 run C1 did
# (v12's forced-PyTorch-conv control showed the implementation choice itself makes
# no difference). The sweep scaffolding (configurable num_blocks / d_state / d_conv,
# the forced-PyTorch-conv switch and its kernel-disabling context manager) is gone.
import sys

MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496      # v12 Stage C winner C1: 2 blocks x 437,248
# The v5-v10 backbone (4 blocks, d_state 16, d_conv 4). Used ONLY to advance the torch RNG
# (see BiModalAeroSyncMamba.__init__) — it is never part of the model.
_V10_BACKBONE_FOR_RNG = (4, 16, 4)


def mamba_conv_path():
    """Which local-conv implementation mamba_ssm will use (informational)."""
    if _MAMBA_CLASS is None:
        return "native (fallback VectorizedSSMBlock nn.Conv1d)"
    if getattr(sys.modules[_MAMBA_CLASS.__module__], "causal_conv1d_fn", None) is not None:
        return "fused (causal-conv1d CUDA kernel)"
    return "pytorch nn.Conv1d (causal-conv1d not importable)"


class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = MAMBA_D_CONV, expand: int = MAMBA_EXPAND,
                 d_state: int = MAMBA_D_STATE):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if _MAMBA_CLASS is not None:
        return nn.ModuleList([
            RealMambaBlock(d_model=d_model, d_conv=d_conv, expand=expand, d_state=d_state)
            for _ in range(num_blocks)
        ])
    # the fallback block has no SSM state, so d_state does not apply to it
    return nn.ModuleList([
        VectorizedSSMBlock(d_model=d_model, d_conv=d_conv, expand=expand) for _ in range(num_blocks)
    ])


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=4, beta=temporal_bias_scale)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        # v13: fixed backbone 2 x (d_state 16, d_conv 2). RNG alignment (NOT a sweep
        # option): the winning v12 run built this stack from the SAME torch RNG state
        # as the v5-v10 4-block stack and then restored the RNG to the state AFTER that
        # 4-block stack. Every module built after this point (pool_attn, classifier,
        # aux heads) and the whole training RNG stream (shuffle order, dropout masks)
        # therefore depend on it. To reproduce the v12 result bit-for-bit, a 4-block
        # stack is built and immediately DISCARDED purely to advance the RNG — it is
        # never registered as a submodule, has no parameters in the model and is never
        # run. Removing it would change every downstream initial weight.
        self.mamba_config = {"num_blocks": MAMBA_NUM_BLOCKS, "d_state": MAMBA_D_STATE,
                             "d_conv": MAMBA_D_CONV, "expand": MAMBA_EXPAND}
        _rng_pre = torch.get_rng_state()
        _rng_advance_only = build_fusion_blocks(d_model, *_V10_BACKBONE_FOR_RNG)
        del _rng_advance_only
        _rng_post = torch.get_rng_state()
        torch.set_rng_state(_rng_pre)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(_rng_post)
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print(f"Bi-Modal AeroSync-Mamba model class defined successfully! | fixed Mamba backbone: {MAMBA_NUM_BLOCKS} blocks, "
      f"d_state={MAMBA_D_STATE}, d_conv={MAMBA_D_CONV}, expand={MAMBA_EXPAND} | conv path: {mamba_conv_path()}")


FUSION BACKBONE RESOLUTION: mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected

Bi-Modal AeroSync-Mamba model class defined successfully! | fixed Mamba backbone: 2 blocks, d_state=16, d_conv=2, expand=2 | conv path: fused (causal-conv1d CUDA kernel)


# Cell 3: Bi-Modal Loss Function (Classification + lambda_align x Audio-IMU InfoNCE + lambda_cons x KL) — all raw components reported

In [3]:
# v13: L = L_cls + lambda_align * L_align + lambda_cons * L_cons. The two weights are set ONLY by
# the constructor arguments below (default 0.1 / 0.1 — the value every experiment v5-v12 used) and stored on
# the module as self.lambda_align / self.lambda_cons; forward() reads them from there. run_condition (Cell 6)
# passes each condition's weights in and verifies them on the constructed module.
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        # v13: every RAW component is reported (loss_cons was previously not), so a term whose weight is
        # 0.0 can still be seen computing a sensible value even though it adds nothing to `total`.
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item()),
                       "loss_cons": float(loss_cons.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

Loss & Metrics helper ready!


# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset

In [4]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

# Three different sizes — kept as separate named constants so they are never confused:
N_AUDIO_BINS = 64             # audio FREQUENCY axis: strongest-64 bins = feature dim per time step (FIXED, v6 winner)
TA_AUDIO = 64                 # audio TIME axis: time positions per window (FIXED, v7)
TS_SENSOR = 128               # sensor TIME axis: time positions per window (FIXED, unchanged from v6)
_audio_chunks_per_window = []  # verification: real audio chunks available in each window (vs Ta)
_MISSION_CACHE = {}  # mission_dir -> parse_real_mission(...) output, parsed ONCE and reused for every N

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


def get_mission(mission_dir: Path):
    """parse_real_mission, memoized: every mission's JSONL is parsed once and the
    same arrays are reused for every N (identical to re-parsing each time)."""
    if mission_dir not in _MISSION_CACHE:
        _MISSION_CACHE[mission_dir] = parse_real_mission(mission_dir)
    return _MISSION_CACHE[mission_dir]


# 2.3 Strongest-N FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once per N, before any train/val/test window slicing for
    that N, so the exact same frozen indices are reused identically across all
    three splits.

    Identical criterion to the v5 strongest-64 selection; only the budget N is
    a parameter now. Returns (sorted bin indices, data-derived bin width in Hz
    or None).
    """
    print(f"[STRONGEST-{n_bins}] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc=f"Bin selection N={n_bins} (train-only audio scan)", leave=False):
        mission_data = get_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        f"STRONGEST-{n_bins} FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-{n_bins} FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )
    bin_width_hz = est_sample_rate_hz / samples_per_buffer if est_sample_rate_hz else None

    print(f"[STRONGEST-{n_bins}] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-{n_bins}] Selected bin indices (ascending): {top_bins.tolist()}")
    if bin_width_hz:
        print(f"[STRONGEST-{n_bins}] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        # contiguous runs of selected bins -> approx. frequency bands
        runs, start = [], top_bins[0]
        for prev, cur in zip(top_bins[:-1], top_bins[1:]):
            if cur != prev + 1:
                runs.append((start, prev))
                start = cur
        runs.append((start, top_bins[-1]))
        print(f"[STRONGEST-{n_bins}] Selected bins as {len(runs)} contiguous band(s) -> approx. frequency (Hz):")
        for lo, hi in runs:
            print(f"    bins {lo:3d}-{hi:3d} ({hi - lo + 1:3d} bins)  ->  {lo * bin_width_hz:8.1f} - {hi * bin_width_hz:8.1f} Hz")
    else:
        print(f"[STRONGEST-{n_bins}] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins, bin_width_hz


# 2.4 Audio feature extraction — the single, permanent path (v5), width = N:
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-N bins -> sqrt
#     (no normalization).
def compute_audio_features(audio_chunks, bin_indices):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    bin_indices  : frozen strongest-N bin indices from select_strongest_audio_bins
                   (computed once per N from TRAINING audio; never recomputed here)
    returns      : (n_chunks, N) float32 = sqrt(|rfft|) at those bins, N = len(bin_indices)
    """
    assert bin_indices is not None and len(bin_indices) > 0, (
        "FEATURES FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )
    n_bins = len(bin_indices)

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = bin_indices[bin_indices < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, n_bins), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, N={n_bins})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, bin_indices, Ta, window_sec=2.56, step_sec=1.28, Ts=128):
    # Ta = audio TIME positions per window (required, no default); Ts = sensor TIME positions (unchanged).
    # The frequency width of every audio time step is n_bins = len(bin_indices), independent of Ta.
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert bin_indices is not None and len(bin_indices) > 0, (
        "SLICE FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest-N audio features: FFT magnitude spectrum, frozen top-N bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-N bins -> sqrt, via compute_audio_features (2.4).
    n_bins = len(bin_indices)
    a_feats = compute_audio_features(audio, bin_indices)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"STRONGEST-{n_bins} FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {n_bins})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-{n_bins} VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-{n_bins} bins) shape: {a_feats.shape}")
        print(f"[STRONGEST-{n_bins} VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              f"(frequency-domain magnitude spectrum) using the frozen strongest-{n_bins} bin indices, "
              f"NOT the first {n_bins} bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        _audio_chunks_per_window.append(int(a_mask.sum()))
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, n_bins), dtype=np.float32)  # (TIME positions, FREQUENCY bins)
        for c in range(n_bins):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, n_bins), (
            f"STRONGEST-{n_bins} FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {n_bins})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats", "N_BUDGETS", "TA_VALUES", "SENSOR_CHANNEL_CONFIGS",
                              "DERIVED_CHANNEL_SOURCES") if n in globals()]

assert _feat_params == ["audio_chunks", "bin_indices"], f"compute_audio_features has unexpected parameters: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v9 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- 5. Strongest-64 bins: selected ONCE from TRAINING missions only, frozen ---
# The FREQUENCY axis does not depend on the audio TIME axis (Ta), so the same
# frozen indices are reused for every Ta condition and for train/val/test.
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
_heldout_missions = {m for mp in (val_mission_map, test_mission_map) for ms in mp.values() for m in ms}
assert not (set(all_train_missions) & _heldout_missions), (
    "LEAKAGE: training missions overlap validation/test missions")

SELECTED_BIN_INDICES, AUDIO_BIN_WIDTH_HZ = select_strongest_audio_bins(all_train_missions, n_bins=N_AUDIO_BINS)
_bin_fingerprint_at_selection = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-{N_AUDIO_BINS}] frozen once from {len(all_train_missions)} TRAINING missions "
      f"(0 of {len(_heldout_missions)} validation/test missions used); reused unchanged for every Ta.")


# --- 6. Per-Ta preprocessing (called once per Ta by run_experiment in Cell 6) ---
def preprocess_for_ta(Ta):
    """Train/val/test windows with Ta audio time positions per window, using the
    frozen strongest-64 bins. Sensor side (TS_SENSOR positions, 6 channels) is
    built by exactly the same code as v6 and does not depend on Ta."""
    global _fft_shape_printed
    _fft_shape_printed = False
    _audio_chunks_per_window.clear()

    split_maps = [("TRAIN", train_mission_map), ("VAL", val_mission_map), ("TEST", test_mission_map)]
    split_samples = {}
    _fp_train = None
    for split_label, mission_map in split_maps:
        note = "(frozen strongest-64 bins)" if split_label == "TRAIN" else "(same frozen bin indices — NOT recomputed)"
        print(f"\n--- Preprocessing {split_label} split, Ta={Ta} {note} ---")
        samples = []
        for c_label in sorted(class_missions.keys()):
            for m in tqdm(mission_map[c_label], desc=f"{split_label.capitalize()} Class {c_label} (Ta={Ta})", leave=False):
                p = get_mission(m)
                if p:
                    samples.extend(slice_real_bimodal(p, c_label, SELECTED_BIN_INDICES, Ta=Ta, Ts=TS_SENSOR))
        split_samples[split_label] = samples

        _fp = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
        if split_label == "TRAIN":
            _fp_train = _fp
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ TRAIN (first5, last5): {_fp} "
                  f"| matches selection: {_fp == _bin_fingerprint_at_selection}")
        else:
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ {split_label:<5} (first5, last5): {_fp} "
                  f"| matches TRAIN: {_fp == _fp_train}")

    # --- Shape verification across all three splits:
    #     audio  (n_windows, Ta, N_AUDIO_BINS)  = (windows, TIME, FREQUENCY)
    #     sensor (n_windows, TS_SENSOR, 6)      — independent of Ta
    sensor_fingerprint = {}
    for split_label, samples in split_samples.items():
        x_a_all = np.stack([s["x_a"] for s in samples])
        tau_a_all = np.stack([s["tau_a"] for s in samples])
        x_s_all = np.stack([s["x_s"] for s in samples])
        tau_s_all = np.stack([s["tau_s"] for s in samples])
        assert x_a_all.shape == (len(samples), Ta, N_AUDIO_BINS), (
            f"Ta={Ta} FAILED: {split_label} audio array is {x_a_all.shape}, "
            f"expected (n_windows={len(samples)}, Ta={Ta}, N_AUDIO_BINS={N_AUDIO_BINS})")
        assert tau_a_all.shape == (len(samples), Ta), f"Ta={Ta} FAILED: {split_label} tau_a is {tau_a_all.shape}"
        assert x_s_all.shape == (len(samples), TS_SENSOR, 6), (
            f"Ta={Ta} FAILED: {split_label} sensor array is {x_s_all.shape}, expected (n_windows, {TS_SENSOR}, 6)")
        assert tau_s_all.shape == (len(samples), TS_SENSOR), f"Ta={Ta} FAILED: {split_label} tau_s is {tau_s_all.shape}"
        sensor_fingerprint[split_label] = (len(samples), round(float(x_s_all.astype(np.float64).sum()), 3),
                                           round(float(tau_s_all.astype(np.float64).sum()), 3),
                                           tuple(np.bincount([s["label"] for s in samples], minlength=5).tolist()))
        print(f"[SHAPE CHECK] {split_label:<5}: audio {x_a_all.shape} = (n_windows, Ta={Ta}, bins={N_AUDIO_BINS}) ✅ | "
              f"sensor {x_s_all.shape} = (n_windows, Ts={TS_SENSOR}, 6) ✅")
        del x_a_all, tau_a_all, x_s_all, tau_s_all

    cpw = np.array(_audio_chunks_per_window)
    n_short = int((cpw < Ta).sum())
    print(f"[AUDIO RESOLUTION] real audio chunks inside a {2.56:.2f}s window: min {cpw.min()} | median "
          f"{int(np.median(cpw))} | max {cpw.max()} -> Ta={Ta} "
          + ("sub-samples the real chunks in every window." if n_short == 0 else
             f"exceeds the available chunks in {n_short}/{len(cpw)} windows ({n_short / len(cpw) * 100:.1f}%), "
             "where some real chunk positions are repeated (no synthetic values are created)."))

    print("\n" + "=" * 60)
    print(f"     100% REAL PREPROCESSING COMPLETE (Ta={Ta}, ZERO SYNTHETIC DATA)   ")
    if _chunk_duration_log:
        _arr = np.array(_chunk_duration_log)
        print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
    print("=" * 60)
    print(f"  • Total Train Diagnostic Windows : {len(split_samples['TRAIN'])}")
    print(f"  • Total Val Diagnostic Windows   : {len(split_samples['VAL'])}")
    print(f"  • Total Test Diagnostic Windows  : {len(split_samples['TEST'])}")
    print("=" * 60)

    out_dir = CACHE_DIR / f"Ta{Ta}"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": out_dir / "bimodal_train.pt", "val": out_dir / "bimodal_val.pt", "test": out_dir / "bimodal_test.pt"}
    torch.save(split_samples["TRAIN"], paths["train"])
    torch.save(split_samples["VAL"], paths["val"])
    torch.save(split_samples["TEST"], paths["test"])
    print(f"✅ Cached real bimodal dataset for Ta={Ta} in {out_dir}")

    return {
        "paths": paths,
        "n_windows": (len(split_samples["TRAIN"]), len(split_samples["VAL"]), len(split_samples["TEST"])),
        "sensor_fingerprint": sensor_fingerprint,
        "chunks_per_window": (int(cpw.min()), int(np.median(cpw)), int(cpw.max())),
        "pct_windows_ta_exceeds_chunks": round(n_short / len(cpw) * 100, 2),
    }

print(f"\n✅ Preprocessing helpers ready — split + strongest-{N_AUDIO_BINS} bins fixed above; window "
      f"preprocessing (Ta = {TA_AUDIO}, Ts = {TS_SENSOR}) is built once in Cell 6 and reused for every beta.")

📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...
Cloning into '/kaggle/working/tii_uav_dataset'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 324 (delta 111), reused 321 (delta 111), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 1.15 GiB | 27.12 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Updating files: 100% (217/217), done.
✅ Dataset cloned successfully!
[NO-MODE-SWITCH CHECK] compute_audio_features('audio_chunks', 'bin_indices') | slice_real_bimodal has no window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif branches: clean ✅
[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅
Class 0 (19 flights) -> Train: 13, Val: 2, Test: 4
Class 1 (20 flights) -> Train: 14, Val: 3, Test: 3
Class 2 (20 flights) -> Tr

Bin selection N=64 (train-only audio scan):   0%|          | 0/69 [00:00<?, ?it/s]

[STRONGEST-64] Selected 64 bins from 356506 training-set audio chunks across 69 missions (FFT length = 769).
[STRONGEST-64] Selected bin indices (ascending): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
[STRONGEST-64] Estimated audio sample rate ≈ 84695.4 Hz (derived from audio buffer length and real inter-buffer timing across training missions; the dataset does not document a mic sample rate directly, so this is a data-derived estimate for labeling only, not a hardcoded spec).
[STRONGEST-64] Selected bins as 7 contiguous band(s) -> approx. frequency (Hz):
    bins   1- 11 ( 11 bins)  ->      55.1 -    606.5 Hz
    bins 245-255 ( 11 bins)  ->   13509.4 -  14060.8 Hz
    bins 258-267 ( 10 bins)  ->   14226.2 -  14722.4 Hz
    bins 501-

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [5]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = int(self.samples[0]["x_a"].shape[0])  # audio TIME positions per window
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

batch_size = 32

def build_dataloaders(paths, batch_size=batch_size):
    """Datasets + loaders (same settings as v7; identical for every beta)."""
    train_ds = RealBiModalDataset(paths["train"])
    val_ds   = RealBiModalDataset(paths["val"], mean=train_ds.mean, std=train_ds.std)
    test_ds  = RealBiModalDataset(paths["test"], mean=train_ds.mean, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

print("RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).")

RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).


In [6]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
def verify_real_taus(train_ds):
    print("=" * 70)
    print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
    print("=" * 70)

    check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
    taus_a, taus_s = [], []
    for idx in check_idxs:
        item = train_ds[idx]
        taus_a.append(item["tau_a"])
        taus_s.append(item["tau_s"])
        print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

    all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
    all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

    assert not all_tau_a_identical, (
        "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    assert not all_tau_s_identical, (
        "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
    print("=" * 70)

print("verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.")

verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.


# Cell 6: run_condition(lambda_align, lambda_cons) — one isolated condition (fixed architecture; only the two auxiliary loss weights vary)

In [7]:
# ==============================================================================
# run_condition(lambda_align, lambda_cons) — ONE complete, isolated condition:
#   seeds reset -> v10 preprocessing (Ta=64, strongest-64, raw 6 IMU; built once,
#   identical for every condition) -> DataLoaders -> FRESH model (beta=1.0, mlp
#   g(Δτ) 97 params, 4 LACA heads, fixed Mamba 2 x d_state 16 x d_conv 2) +
#   class-weighted loss with THIS condition's (lambda_align, lambda_cons) + FRESH
#   optimizer + scheduler -> 20-epoch training with best-val checkpoint ->
#   unseen-test evaluation (live forward pass, + learned lag Δ_as) -> GPU memory freed.
#
# Where the two loss weights live (Cell 3):
#   BiModalAeroSyncLoss(lambda_align=..., lambda_cons=...) stores them as
#   criterion.lambda_align / criterion.lambda_cons, and forward() computes
#     total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
#   v5-v12 hard-coded BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1, ...) in
#   the runner; here they are this function's two arguments and nothing else changes.
# ==============================================================================
import gc
import hashlib

EPOCHS = 20
D_MODEL = 256
_PREP = None   # v10 preprocessing result (Ta=64), built once and reused by every condition

# Torch CPU RNG state right after model construction, printed by v10 ("[INIT]" line). v12 confirmed the winning
# 2 x (16, 2) run left the RNG in exactly this state (RNG identical to its 4-block baseline), so it must match here.
V10_RNG_FP_PREFIX = "5641b401720d"


def _sha1_state(model, exclude):
    h = hashlib.sha1()
    for k, v in model.state_dict().items():
        if any(e in k for e in exclude):
            continue
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def init_fingerprints(model):
    """full initial state (must be identical across every loss-weight condition — the weights do not touch the
    model), and the torch CPU RNG state right after construction."""
    return {
        "init": _sha1_state(model, ()),
        "rng": hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest(),
    }


def lag_bias_param_arithmetic(g):
    """(exact learnable-parameter count, hand-checkable arithmetic) for the Linear/Tanh/Linear g(Δτ)."""
    terms = []
    for m in g.modules():
        if isinstance(m, nn.Linear):
            terms.append(f"Linear({m.in_features},{m.out_features}) weights {m.weight.numel()}")
            terms.append(f"bias {m.bias.numel()}")
    total = sum(p.numel() for p in g.parameters())
    return total, " + ".join(terms) + f" = {total} params"


def verify_lag_bias(model):
    """the mlp g(Δτ) is the original v5-v9 Linear(1,32)-Tanh-Linear(32,1), 97 params, 4 heads, beta 1.0."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        g = laca.lag_bias
        n, arith = lag_bias_param_arithmetic(g)
        print(f"[LAG-BIAS] model.laca.{name}.lag_bias: {arith} | num_heads={laca.num_heads} | beta={laca.beta}")
        assert (isinstance(g, nn.Sequential) and len(g) == 3 and isinstance(g[1], nn.Tanh)
                and (g[0].in_features, g[0].out_features, g[2].in_features, g[2].out_features) == (1, 32, 32, 1)), (
            f"{name}.lag_bias is not the original mlp: {g}")
        assert n == 97, f"{name}.lag_bias has {n} params, expected 97"
        assert laca.num_heads == 4 and laca.beta == LACA_BETA


def mamba_param_counts(model):
    per_block = [sum(p.numel() for p in b.parameters()) for b in model.mamba_blocks]
    return sum(per_block), per_block


def verify_mamba_backbone(model):
    """Confirm, from the CONSTRUCTED blocks (not the constants), that the fixed v12-winner backbone is in use,
    and that its parameter count matches the known winning run (874,496)."""
    blocks = model.mamba_blocks
    assert len(blocks) == MAMBA_NUM_BLOCKS, f"{len(blocks)} Mamba blocks built, expected {MAMBA_NUM_BLOCKS}"
    total, per_block = mamba_param_counts(model)
    b0 = blocks[0]
    if isinstance(b0, RealMambaBlock):
        for i, b in enumerate(blocks):
            m = b.mamba
            assert (m.d_state, m.d_conv, m.expand, m.d_model) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND, D_MODEL), (
                f"block {i}: d_state={m.d_state} d_conv={m.d_conv} expand={m.expand} d_model={m.d_model}")
            assert m.conv1d.kernel_size == (MAMBA_D_CONV,), f"block {i}: conv kernel {m.conv1d.kernel_size}"
            assert tuple(m.A_log.shape) == (m.d_inner, MAMBA_D_STATE), f"block {i}: A_log {tuple(m.A_log.shape)}"
        m = b0.mamba
        print(f"[MAMBA CHECK] {len(blocks)} x RealMambaBlock | d_state={m.d_state} (A_log {tuple(m.A_log.shape)}) | "
              f"d_conv={m.d_conv} (conv1d kernel {m.conv1d.kernel_size[0]}) | expand={m.expand} | d_inner={m.d_inner} | "
              f"local-conv path: {mamba_conv_path()}")
        print(f"[MAMBA PARAMS] fusion backbone total = {total:,} ({len(per_block)} blocks x {per_block[0]:,}) | "
              f"known winning v12 run = {EXPECTED_MAMBA_PARAMS:,} -> "
              f"{'MATCH ✅' if total == EXPECTED_MAMBA_PARAMS else 'MISMATCH ❌'} | "
              f"whole model = {sum(p.numel() for p in model.parameters()):,}")
        assert total == EXPECTED_MAMBA_PARAMS, (
            f"Mamba backbone has {total:,} params, the v12 winner had {EXPECTED_MAMBA_PARAMS:,}")
    else:
        print(f"[MAMBA CHECK] 🚨 {len(blocks)} x VectorizedSSMBlock (FALLBACK — no SSM state; d_state does not apply) | "
              f"d_conv={b0.conv1d.kernel_size[0]} | {total:,} params — NOT the real Mamba backbone, so the "
              f"{EXPECTED_MAMBA_PARAMS:,}-param check and the Part 0 reproduction CANNOT pass on this machine.")
    return total, per_block[0]


def verify_shapes(model, batch_t, when):
    """Real batch through the model's OWN modules; assert every shape the fusion step relies on.
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated."""
    xa, xs, ta, ts = batch_t
    was_training = model.training
    model.eval()
    with torch.no_grad():
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, modality_idx=0)
        Hs = model.htt(zs, ts, modality_idx=1)
        B, Ta_, Ts_ = Ha.shape[0], Ha.shape[1], Hs.shape[1]
        H_tilde_a, H_tilde_s, aux = model.laca(Ha, Hs, ta, ts)
        assert H_tilde_a.shape == (B, Ta_, D_MODEL) and H_tilde_s.shape == (B, Ts_, D_MODEL)
        F_seq = torch.cat([H_tilde_a, model.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        L = Ta_ + 1 + Ts_
        assert F_seq.shape == (B, L, D_MODEL), f"Mamba input {tuple(F_seq.shape)}"
        blk0_in = F_seq
        for i, blk in enumerate(model.mamba_blocks):
            F_seq = blk(F_seq)
            assert F_seq.shape == (B, L, D_MODEL) and torch.isfinite(F_seq).all(), (
                f"Mamba block {i} output {tuple(F_seq.shape)} (finite={bool(torch.isfinite(F_seq).all())})")
        logits, pooled, aux_preds, aux2 = model(xa, xs, ta, ts)
        assert logits.shape == (B, model.n_classes) and torch.isfinite(logits).all(), f"logits {tuple(logits.shape)}"
        assert pooled["z_a"].shape == pooled["z_s"].shape == (B, D_MODEL)
        assert aux2["lag_as"].shape == aux2["lag_sa"].shape == (B,)
        print(f"[SHAPES {when}] Mamba input {tuple(blk0_in.shape)} -> {len(model.mamba_blocks)} blocks, each "
              f"{(B, L, D_MODEL)} ✅ -> logits {tuple(logits.shape)} ✅")
    model.train(was_training)


def verify_loss_weights(criterion, lambda_align, lambda_cons, model, batch_t, labels_t):
    """Confirm the loss weights from the CONSTRUCTED loss module, not from the config variables:
      1. read criterion.lambda_align / criterion.lambda_cons and compare with the requested values;
      2. run the module on a real batch and check that the total it returns equals
         loss_cls + criterion.lambda_align * loss_align + criterion.lambda_cons * loss_cons,
         i.e. the stored weights are the ones actually applied inside forward().
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated, no gradients."""
    la, lc = criterion.lambda_align, criterion.lambda_cons
    print(f"[LOSS WEIGHTS] requested: lambda_align={lambda_align} | lambda_cons={lambda_cons}")
    print(f"[LOSS WEIGHTS] read from the constructed {type(criterion).__name__}: "
          f"criterion.lambda_align={la!r} | criterion.lambda_cons={lc!r}")
    assert float(la) == float(lambda_align) and float(lc) == float(lambda_cons), (
        f"loss module holds ({la}, {lc}) but ({lambda_align}, {lambda_cons}) was requested")
    was_training = model.training
    model.eval()
    with torch.no_grad():
        logits, pooled, aux_preds, _ = model(*batch_t)
        total, comp = criterion(logits, labels_t, pooled, aux_preds)
    model.train(was_training)
    total = float(total.item())
    recon = comp["loss_cls"] + la * comp["loss_align"] + lc * comp["loss_cons"]
    naive = comp["loss_cls"] + 0.1 * comp["loss_align"] + 0.1 * comp["loss_cons"]
    print(f"[LOSS WEIGHTS] functional check on a real batch: total={total:.6f} | loss_cls={comp['loss_cls']:.6f} | "
          f"loss_align={comp['loss_align']:.6f} (x{la} -> {la * comp['loss_align']:.6f}) | "
          f"loss_cons={comp['loss_cons']:.6f} (x{lc} -> {lc * comp['loss_cons']:.6f})")
    assert all(math.isfinite(v) for v in comp.values()), f"non-finite loss component: {comp}"
    assert abs(total - recon) <= 1e-4 * max(1.0, abs(recon)), (
        f"total {total:.6f} != cls + {la}*align + {lc}*cons = {recon:.6f} — the module is NOT applying these weights")
    if (la, lc) != (0.1, 0.1):
        # the default-weight total must NOT be what forward() returned (otherwise the weights were ignored)
        assert abs(total - naive) > 1e-6, "total equals the 0.1/0.1 total — the new weights were not applied"
    for name, w in (("loss_align", la), ("loss_cons", lc)):
        if w == 0.0:
            print(f"[LOSS WEIGHTS]   {name}: weight 0.0 -> still COMPUTED (raw {comp[name]:.6f}, finite) but adds "
                  f"0.0 to the total")
    print(f"[LOSS WEIGHTS] ✅ constructed loss module applies lambda_align={la}, lambda_cons={lc} "
          f"(total == cls + {la}*align + {lc}*cons to within 1e-4)")


def run_condition(lambda_align, lambda_cons, tag=None, seed=SEED, epochs=EPOCHS):
    global _PREP
    t_run0 = time.time()
    tag = tag or f"la={lambda_align},lc={lambda_cons}"
    set_all_seeds(seed)
    print(f"[SEEDS] reset at condition start -> random / numpy / torch / cuda = {seed}")
    print(f"[CONDITION] {tag} | lambda_align={lambda_align} | lambda_cons={lambda_cons} | fixed: beta={LACA_BETA}, "
          f"mlp g(Δτ), 4 heads, Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV})")

    # --- 1. v10 preprocessing (Ta=64) — identical for every condition, built on first use ---
    if _PREP is None:
        _PREP = preprocess_for_ta(TA_AUDIO)
    else:
        print(f"[DATA] reusing the Ta={TA_AUDIO} windows built for the first condition (identical data for every condition)")
    prep = _PREP

    # --- 2. Datasets / loaders (v10 settings) ---
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_dataloaders(prep["paths"])
    verify_real_taus(train_ds)

    # Batch built straight from the dataset (no DataLoader iteration -> no RNG consumed).
    probe = [train_ds[i] for i in range(min(batch_size, len(train_ds)))]
    probe_t = tuple(torch.stack([p[k] for p in probe]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    probe_labels = torch.stack([p["label"] for p in probe]).to(device)
    assert tuple(probe_t[0].shape) == (len(probe), TA_AUDIO, N_AUDIO_BINS)
    assert tuple(probe_t[1].shape) == (len(probe), TS_SENSOR, 6)

    # --- 3. FRESH model (seed reset right before, exactly as v5-v12) ---
    set_all_seeds(seed)
    model = BiModalAeroSyncMamba(
        d_model=D_MODEL,
        n_classes=5,
        dropout=0.15,
        temporal_bias_scale=LACA_BETA,        # FIXED at 1.0
        audio_in_dim=N_AUDIO_BINS,
        sensor_in_dim=6,
    ).to(device)
    fps = init_fingerprints(model)

    print("\n" + "-" * 90)
    verify_lag_bias(model)
    mamba_total, mamba_per_block = verify_mamba_backbone(model)
    rng_matches_v12 = fps["rng"].startswith(V10_RNG_FP_PREFIX)
    print(f"[INIT] torch RNG after construction {fps['rng'][:12]} vs the v12 winning run ({V10_RNG_FP_PREFIX}): "
          + ("IDENTICAL ✅ (same shuffle order + dropout masks as v12 C1)" if rng_matches_v12 else
             "DIFFERENT ⚠️  (RNG stream differs from v12; a different torch build can also cause this)"))
    ref = _BASELINE_FPS if "_BASELINE_FPS" in globals() else None
    if ref is not None:
        same_init, same_rng = fps["init"] == ref["init"], fps["rng"] == ref["rng"]
        print(f"[INIT] every initial weight identical to the Part 0 baseline: {'YES ✅' if same_init else 'NO ❌'}"
              f" | RNG state after construction identical: {'YES ✅' if same_rng else 'NO ❌'}")
        if not (same_init and same_rng):
            print("🚨" * 40 + "\n🚨  the model / RNG stream differs from the baseline — this condition does NOT isolate "
                  "the loss weights.\n" + "🚨" * 40)
    else:
        same_init = same_rng = None
    verify_shapes(model, probe_t, "before training")

    _fusion_cls = type(model.mamba_blocks[0])
    print(f"[MODEL] FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}"
          + ("  🚨 pure-PyTorch FALLBACK, NOT real mamba_ssm 🚨" if _fusion_cls.__name__ == "VectorizedSSMBlock"
             else "  ✅ REAL mamba_ssm selective-scan SSM"))

    # FIX 4: inverse-frequency class weights from REAL training-set class counts
    train_labels_arr = np.array([s["label"] for s in train_ds.samples])
    class_counts = np.bincount(train_labels_arr, minlength=5)
    class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
    print("[FIX 4] class weights (from real train counts): "
          + " | ".join(f"C{c}: n={class_counts[c]} w={class_weights_np[c]:.4f}" for c in range(5)))

    # --- the ONLY thing that changes between conditions: the two auxiliary loss weights ---
    criterion = BiModalAeroSyncLoss(lambda_align=lambda_align, lambda_cons=lambda_cons, class_weights=class_weights)
    verify_loss_weights(criterion, lambda_align, lambda_cons, model, probe_t, probe_labels)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    print("-" * 90 + "\n")

    best_val_f1 = 0.0
    best_weights = None
    best_epoch = None
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": [],
               "train_loss_cls": [], "train_loss_align": [], "train_loss_cons": [],
               "val_loss_cls": [], "val_loss_align": [], "val_loss_cons": []}
    ckpt_path = CACHE_DIR / f"aerosync_bimodal_best_la{lambda_align}_lc{lambda_cons}.pt"

    try:
        # --- 4. Training (identical loop to v5-v12; the extra component sums only read .item() values) ---
        start_time = time.time()
        print(f"🚀 Training Bi-Modal AeroSync-Mamba ({tag}) on {len(train_ds)} real flight windows...\n")

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, running_cls = 0.0, 0.0
            running_align, running_cons = 0.0, 0.0

            pbar = tqdm(train_loader, desc=f"{tag} Epoch {epoch:02d}/{epochs}", leave=False)
            for batch in pbar:
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer.zero_grad()
                logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                running_cls += l_dict["loss_cls"]
                running_align += l_dict["loss_align"]
                running_cons += l_dict["loss_cons"]
                pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

            scheduler.step()
            n_tr = len(train_loader)
            avg_train_loss = running_loss / n_tr

            # --- Validation ---
            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            val_loss_total = 0.0
            val_comp = {"loss_cls": 0.0, "loss_align": 0.0, "loss_cons": 0.0}

            with torch.no_grad():
                for batch in val_loader:
                    xa = batch["x_a"].to(device)
                    xs = batch["x_s"].to(device)
                    ta = batch["tau_a"].to(device)
                    ts = batch["tau_s"].to(device)
                    labels = batch["label"].to(device)

                    logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                    v_loss, v_dict = criterion(logits, labels, pooled_reps, aux_preds)
                    val_loss_total += v_loss.item()
                    for k in val_comp:
                        val_comp[k] += v_dict[k]

                    probs = F.softmax(logits, dim=-1)
                    val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    val_targets.extend(labels.cpu().numpy())
                    val_probs.extend(probs.cpu().numpy())

            avg_val_loss = val_loss_total / len(val_loader)
            val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["val_f1"].append(val_metrics["macro_f1"])
            history["val_acc"].append(val_metrics["accuracy"])
            history["train_loss_cls"].append(running_cls / n_tr)
            history["train_loss_align"].append(running_align / n_tr)
            history["train_loss_cons"].append(running_cons / n_tr)
            for k in val_comp:
                history[f"val_{k}"].append(val_comp[k] / len(val_loader))

            is_best = ""
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                torch.save(best_weights, ckpt_path)
                is_best = " ⭐ [Best Saved]"

            print(
                f"{tag} Epoch {epoch:02d}/{epochs} | "
                f"Train Loss: {avg_train_loss:.4f} (cls {running_cls / n_tr:.4f} | align {running_align / n_tr:.4f} | "
                f"cons {running_cons / n_tr:.4f}) | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
            )

        train_seconds = time.time() - start_time
        print(f"\n🎉 {tag} Training Finished in {train_seconds:.1f}s | Best Val Macro-F1: {best_val_f1:.4f} "
              f"(epoch {best_epoch})")

        # --- Final-epoch loss components (RAW values, before weighting) ---
        fe = {k: history[k][-1] for k in history}
        print(f"[LOSS COMPONENTS @ final epoch {epochs}] (lambda_align={criterion.lambda_align}, "
              f"lambda_cons={criterion.lambda_cons})")
        for split in ("train", "val"):
            print(f"   {split:<5}: loss_cls {fe[f'{split}_loss_cls']:.4f} | loss_align {fe[f'{split}_loss_align']:.4f} "
                  f"(weighted {criterion.lambda_align * fe[f'{split}_loss_align']:.4f}) | loss_cons "
                  f"{fe[f'{split}_loss_cons']:.4f} (weighted {criterion.lambda_cons * fe[f'{split}_loss_cons']:.4f}) | "
                  f"total {fe[f'{split}_loss']:.4f}")
        for name, w in (("loss_align", criterion.lambda_align), ("loss_cons", criterion.lambda_cons)):
            if w == 0.0:
                print(f"   ↪ {name} has weight 0.0: its raw value above is still computed every step but contributes "
                      f"nothing to the total/gradient (epoch-1 train value {history['train_' + name][0]:.4f} -> "
                      f"final {fe['train_' + name]:.4f}).")

        # --- 5. Unseen test evaluation with the best-val weights — LIVE forward pass (as every prior experiment) ---
        if best_weights is not None:
            model.load_state_dict(best_weights)
        model.eval()
        verify_shapes(model, probe_t, "best checkpoint")

        test_preds, test_targets, test_probs, lags_as_list, lags_sa_list = [], [], [], [], []
        n_test_batches = 0
        with torch.no_grad():
            for batch in test_loader:
                xa = batch["x_a"].to(device)
                xs = batch["x_s"].to(device)
                ta = batch["tau_a"].to(device)
                ts = batch["tau_s"].to(device)
                labels = batch["label"].to(device)

                logits, _, _, aux = model(xa, xs, ta, ts)
                probs = F.softmax(logits, dim=-1)
                test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                test_targets.extend(labels.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())
                lags_as_list.extend(aux["lag_as"].cpu().numpy())
                lags_sa_list.extend(aux["lag_sa"].cpu().numpy())
                n_test_batches += 1

        n_test = len(test_targets)
        # provenance: every test window went through THIS model in THIS call, scored by the same
        # compute_all_metrics (Cell 3) used by every prior experiment
        assert n_test_batches == len(test_loader) and n_test == len(test_ds) == len(test_preds) == len(test_probs), (
            f"test pass incomplete: {n_test_batches}/{len(test_loader)} batches, {n_test}/{len(test_ds)} windows")
        test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs))
        n_correct = int((np.array(test_targets) == np.array(test_preds)).sum())
        lag_as_mean = float(np.mean(lags_as_list))
        lag_as_std = float(np.std(lags_as_list))
        lag_sa_mean = float(np.mean(lags_sa_list))
        print("=" * 65)
        print(f"     BI-MODAL AEROSYNC-MAMBA ({tag}): 100% REAL TEST BENCHMARK")
        print("=" * 65)
        print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
        print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
        print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
        print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
        print(f"  • Learned Audio-IMU Lag (Δ_as) : {lag_as_mean:+.4f} (normalized units) "
              f"≈ {lag_as_mean * WINDOW_SEC * 1000:+.1f} ms | std over test windows {lag_as_std:.4f}")
        print(f"  • Learned IMU-Audio Lag (Δ_sa) : {lag_sa_mean:+.4f} (normalized units)")
        print(f"  • Loss weights        : lambda_align={criterion.lambda_align}, lambda_cons={criterion.lambda_cons}")
        print(f"  • Metrics source      : LIVE forward pass, best-val checkpoint (epoch {best_epoch}), "
              f"{n_test} test windows / {n_test_batches} batches")
        print("=" * 65)
    finally:
        # --- 6. Free GPU memory so sequential conditions never accumulate ---
        del model, optimizer, scheduler, criterion, best_weights, probe_t, probe, probe_labels
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[GPU] freed | allocated={torch.cuda.memory_allocated() / 1024**2:.1f} MB "
                  f"| reserved={torch.cuda.memory_reserved() / 1024**2:.1f} MB")

    return {
        "lambda_align": float(lambda_align),
        "lambda_cons": float(lambda_cons),
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "n_test_correct": n_correct,
        "final_train_loss": round(fe["train_loss"], 6),
        "final_train_loss_cls": round(fe["train_loss_cls"], 6),
        "final_train_loss_align": round(fe["train_loss_align"], 6),
        "final_train_loss_cons": round(fe["train_loss_cons"], 6),
        "final_val_loss_cls": round(fe["val_loss_cls"], 6),
        "final_val_loss_align": round(fe["val_loss_align"], 6),
        "final_val_loss_cons": round(fe["val_loss_cons"], 6),
        "learned_lag_as_mean": round(lag_as_mean, 6),
        "learned_lag_as_std": round(lag_as_std, 6),
        "learned_lag_as_ms": round(lag_as_mean * WINDOW_SEC * 1000, 2),
        "learned_lag_sa_mean": round(lag_sa_mean, 6),
        "best_val_macro_f1": round(float(best_val_f1), 6),
        "best_epoch": best_epoch,
        "mamba_params": mamba_total,
        "conv_path": mamba_conv_path(),
        "rng_matches_v12": rng_matches_v12,
        "init_matches_baseline": same_init,
        "rng_matches_baseline": same_rng,
        "n_test_windows": n_test,
        "metrics_source": f"live forward pass (best-val checkpoint epoch {best_epoch}, {n_test} test windows)",
        "seed": seed,
        "train_seconds": round(train_seconds, 1),
        "run_seconds": round(time.time() - t_run0, 1),
        # reproduction fingerprints (not written to the CSV)
        "_fps": fps,
        "_n_windows": prep["n_windows"],
        "_class_counts": class_counts.tolist(),
        "_sensor_fingerprint": prep["sensor_fingerprint"],
        "_epoch1": {"train_loss": history["train_loss"][0], "val_loss": history["val_loss"][0],
                    "val_acc": history["val_acc"][0], "val_f1": history["val_f1"][0]},
    }

WINDOW_SEC = 2.56   # slice_real_bimodal window length; tau is normalized by it in RealBiModalDataset
print(f"run_condition(lambda_align, lambda_cons) ready | fixed Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, "
      f"d_conv {MAMBA_D_CONV}), conv path: {mamba_conv_path()} | fixed: beta={LACA_BETA}, mlp g(Δτ) 97 params, "
      f"4 LACA heads, Ta={TA_AUDIO}, N={N_AUDIO_BINS}, Ts={TS_SENSOR}")


run_condition(lambda_align, lambda_cons) ready | fixed Mamba 2 x (d_state 16, d_conv 2), conv path: fused (causal-conv1d CUDA kernel) | fixed: beta=1.0, mlp g(Δτ) 97 params, 4 LACA heads, Ta=64, N=64, Ts=128


# Cell 7: PART 0 — consolidated Mamba baseline must reproduce the v12 winner (acc 0.9080, macro-F1 0.9048, bal-acc 0.9022, AUROC 0.9894) before Part 1

In [8]:
# ==============================================================================
# PART 0 — the consolidated Mamba backbone (2 blocks x d_state 16 x d_conv 2, fused)
# with the default loss weights (lambda_align=0.1, lambda_cons=0.1) must reproduce
# the v12 winning run BEFORE Part 1 runs. This run is also Part 1's "baseline"
# condition (reused there, not re-trained).
# Results CSV + reuse bookkeeping for the whole notebook live here too.
# ==============================================================================
import csv
import traceback
from datetime import datetime

RESULTS_CSV = Path("/kaggle/working/loss_weight_ablation_results.csv")
RESULTS_CSV_FIELDS = ["stage", "condition_id", "lambda_align", "lambda_cons",
                      "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "n_test_correct",
                      "final_train_loss", "final_train_loss_cls", "final_train_loss_align", "final_train_loss_cons",
                      "final_val_loss_cls", "final_val_loss_align", "final_val_loss_cons",
                      "learned_lag_as_mean", "learned_lag_as_std", "learned_lag_as_ms", "learned_lag_sa_mean",
                      "best_val_macro_f1", "best_epoch", "mamba_params", "conv_path", "rng_matches_v12",
                      "init_matches_baseline", "rng_matches_baseline", "n_test_windows", "metrics_source",
                      "status", "seed", "train_seconds", "run_seconds", "note", "finished_at"]

# If True, Parts 1-2 do NOT run when Part 0 fails to reproduce the v12 winning run.
REQUIRE_BASELINE_REPRODUCTION = True

DEFAULT_LAMBDAS = (0.1, 0.1)   # (lambda_align, lambda_cons) — the value every experiment v5-v12 used

# NEW known baseline (v12 Stage C winner C1: Mamba 2 x d_state 16 x d_conv 2, fused) — supersedes the
# pre-sweep v10 numbers (macro-F1 0.8966) for every sanity check in this notebook.
V12_BASELINE = {"accuracy": 0.9080, "macro_f1": 0.9048, "balanced_accuracy": 0.9022, "macro_auroc": 0.9894}
V12_LAG_AS = -0.0291
V12_BEST_EPOCH = 17
V12_EPOCH1 = {"train_loss": 1.4173, "val_loss": 0.8880, "val_acc": 0.8197, "val_f1": 0.8280}
# Deterministic fingerprints of the v7-v12 data path — independent of GPU numerics, must match EXACTLY.
V10_BIN_INDICES_N64 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                       258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                       509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                       761, 762, 763, 764, 765, 766]
V10_N_WINDOWS = (4934, 954, 1130)
V10_CLASS_COUNTS = [906, 1020, 1004, 1008, 996]
V10_SENSOR_FINGERPRINT = {
    "TRAIN": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
    "VAL":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
    "TEST":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
}
REPRO_TOL = 0.0005

_RESULTS_BY_LAMBDAS = {}   # (lambda_align, lambda_cons) -> finished computed row, for reuse


def append_row(row):
    out = {k: row.get(k) for k in RESULTS_CSV_FIELDS}
    out["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(out)
    print(f"💾 appended {row['stage']} / {row['condition_id']} to {RESULTS_CSV} ({'header + row' if write_header else 'row'})")


def assert_metrics_provenance(row):
    """Final metrics must come from a live forward pass over the full test set (as every prior experiment),
    unless the row is EXPLICITLY marked as reused from an identical, already-computed configuration."""
    if row["status"] == "computed":
        assert str(row["metrics_source"]).startswith("live forward pass"), (
            f"{row['stage']}/{row['condition_id']}: computed row without a live-forward-pass metrics source")
        assert int(row["n_test_windows"]) == V10_N_WINDOWS[2], (
            f"{row['stage']}/{row['condition_id']}: scored on {row['n_test_windows']} test windows, "
            f"expected {V10_N_WINDOWS[2]}")
    else:
        assert str(row["status"]).startswith("reused from") and str(row["metrics_source"]).startswith("REUSED"), (
            f"{row['stage']}/{row['condition_id']}: metrics not from a live pass and not explicitly marked as reused")
    return True


def run_or_reuse(stage, condition_id, lambda_align, lambda_cons, note=""):
    """Either reuse an identical already-run (lambda_align, lambda_cons) condition (same rule as every earlier
    sweep) or train it. The row is appended to the CSV right away either way."""
    key = (float(lambda_align), float(lambda_cons))
    print(f"[CONFIG] {stage}/{condition_id}: lambda_align={lambda_align} | lambda_cons={lambda_cons}")
    if key in _RESULTS_BY_LAMBDAS:
        prev = _RESULTS_BY_LAMBDAS[key]
        print(f"♻️  REUSED, NOT RE-RUN: {stage}/{condition_id} has loss weights identical to already-computed "
              f"{prev['stage']}/{prev['condition_id']} -> copying its metrics (macro_f1={prev['macro_f1']:.4f}).")
        print(f"[LOSS COMPONENTS @ final epoch] (reused from {prev['stage']}/{prev['condition_id']}) train: "
              f"loss_cls {prev['final_train_loss_cls']:.4f} | loss_align {prev['final_train_loss_align']:.4f} | "
              f"loss_cons {prev['final_train_loss_cons']:.4f}")
        row = {k: v for k, v in prev.items() if not k.startswith("_")}
        row.update({"stage": stage, "condition_id": condition_id, "train_seconds": 0.0, "run_seconds": 0.0,
                    "status": f"reused from {prev['stage']}/{prev['condition_id']} (not re-run)",
                    "metrics_source": f"REUSED from {prev['stage']}/{prev['condition_id']} ({prev['metrics_source']})",
                    "note": note})
        assert_metrics_provenance(row)
        append_row(row)
        return row
    res = run_condition(lambda_align, lambda_cons, tag=f"{stage}/{condition_id} la={lambda_align},lc={lambda_cons}")
    res.update({"stage": stage, "condition_id": condition_id, "status": "computed", "note": note})
    assert_metrics_provenance(res)
    _RESULTS_BY_LAMBDAS[key] = res
    append_row(res)
    return res


def check_baseline_against_v12(res):
    """Loud reproduction check for the consolidated baseline. Returns (ok, one-line verdict)."""
    print("\n" + "=" * 90)
    print("  PART 0 CHECK — does the consolidated Mamba 2 x (d_state 16, d_conv 2) baseline reproduce the v12 winner?")
    print("=" * 90)
    det = {
        "strongest-64 bin indices": SELECTED_BIN_INDICES.tolist() == V10_BIN_INDICES_N64,
        f"train/val/test windows {V10_N_WINDOWS}": tuple(res["_n_windows"]) == V10_N_WINDOWS,
        f"train class counts {V10_CLASS_COUNTS}": res["_class_counts"] == V10_CLASS_COUNTS,
        "raw sensor values/timestamps/labels (v10 fingerprint)": res["_sensor_fingerprint"] == V10_SENSOR_FINGERPRINT,
        f"Mamba backbone params = {EXPECTED_MAMBA_PARAMS:,}": int(res["mamba_params"]) == EXPECTED_MAMBA_PARAMS,
    }
    for name, ok in det.items():
        print(f"  [deterministic] {name:<53}: {'MATCH ✅' if ok else 'MISMATCH ❌'}")
    print(f"  [init]          {'torch RNG after construction vs v12 (' + V10_RNG_FP_PREFIX + ')':<53}: "
          f"{'MATCH ✅' if res['rng_matches_v12'] else 'MISMATCH ⚠️'}")
    e1 = res["_epoch1"]
    print("  [epoch 1]       " + " | ".join(
        f"{k}: v12 {V12_EPOCH1[k]:.4f} vs now {e1[k]:.4f} ({e1[k] - V12_EPOCH1[k]:+.4f})" for k in V12_EPOCH1))

    print(f"\n  {'metric':<20} {'v12 known':>10} {'this run':>10} {'delta':>10}")
    worst = 0.0
    for k, v in V12_BASELINE.items():
        d = res[k] - v
        worst = max(worst, abs(d))
        print(f"  {k:<20} {v:>10.4f} {res[k]:>10.4f} {d:>+10.4f}{'' if abs(d) <= REPRO_TOL else '   <-- differs'}")
    print(f"  {'learned lag Δ_as':<20} {V12_LAG_AS:>+10.4f} {res['learned_lag_as_mean']:>+10.4f} "
          f"{res['learned_lag_as_mean'] - V12_LAG_AS:>+10.4f}   (v12 C1 value; informational)")
    print(f"  {'best epoch':<20} {V12_BEST_EPOCH:>10} {res['best_epoch']:>10}   (informational)")

    if not all(det.values()):
        ok, verdict = False, "PORT BUG: deterministic data / backbone fingerprint mismatch"
        print("\n" + "🚨" * 40)
        print("🚨  PART 0 FAILED — bins / windows / class counts / sensor values / Mamba param count differ from v12,")
        print("🚨  which cannot be GPU noise: the data path or the backbone changed in the v12 -> v13 consolidation.")
        print("🚨" * 40)
    elif worst <= REPRO_TOL:
        ok, verdict = True, "reproduces v12 winner (all 4 metrics within 0.0005)"
        print(f"\n✅ PART 0 PASSED: the consolidated baseline REPRODUCES the v12 winner — all four test metrics within "
              f"±{REPRO_TOL} and all deterministic fingerprints match.")
    else:
        ok, verdict = False, f"DOES NOT reproduce v12 metrics (max |delta| {worst:.4f}); data fingerprints match"
        print("\n" + "🚨" * 40)
        print(f"🚨  PART 0 FAILED — TEST METRICS DIFFER FROM THE v12 WINNER (largest |delta| = {worst:.4f} > {REPRO_TOL}).")
        print("🚨  Data path is identical, so the difference arose in training. Check the [init] and epoch-1 lines:")
        print("🚨  if the RNG or epoch 1 already differ, the model/RNG path changed in the consolidation; if they match")
        print("🚨  and later epochs drift, it is GPU non-determinism.")
        print("🚨" * 40)
    print("=" * 90)
    return ok, verdict


if RESULTS_CSV.exists():
    _backup = RESULTS_CSV.with_suffix(f".{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak.csv")
    RESULTS_CSV.rename(_backup)
    print(f"ℹ️  existing results file moved to {_backup}")

_sweep_t0 = time.time()
BASELINE_ROW, BASELINE_OK, BASELINE_VERDICT = None, False, "not run"
print("\n\n" + "=" * 90)
print(f"=== PART 0: consolidated Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss weights (lambda_align, lambda_cons) = {DEFAULT_LAMBDAS} ===")
print("=" * 90)

# --- restored / final Mamba parameter count, from a throwaway construction (RNG saved + restored around it) ---
_rng_saved = torch.get_rng_state()
_probe_model = BiModalAeroSyncMamba(d_model=D_MODEL, n_classes=5, dropout=0.15, temporal_bias_scale=LACA_BETA,
                                    audio_in_dim=N_AUDIO_BINS, sensor_in_dim=6)
_probe_total, _probe_per_block = mamba_param_counts(_probe_model)
del _probe_model
torch.set_rng_state(_rng_saved)
print(f"[PART 0] restored / final Mamba parameter count: {_probe_total:,} ({MAMBA_NUM_BLOCKS} blocks x "
      f"{_probe_per_block[0]:,}) | known winning v12 run C1: {EXPECTED_MAMBA_PARAMS:,} -> "
      + ("MATCH ✅" if _probe_total == EXPECTED_MAMBA_PARAMS else
         "MISMATCH ❌" + (" (FALLBACK backbone — no mamba_ssm)" if _MAMBA_CLASS is None else "")))

try:
    set_all_seeds(SEED)
    BASELINE_ROW = run_condition(*DEFAULT_LAMBDAS, tag="P0/baseline la=0.1,lc=0.1")
    BASELINE_ROW.update({"stage": "P0", "condition_id": "baseline", "status": "computed"})
    assert_metrics_provenance(BASELINE_ROW)
    _BASELINE_FPS = BASELINE_ROW["_fps"]
    _RESULTS_BY_LAMBDAS[DEFAULT_LAMBDAS] = BASELINE_ROW
    BASELINE_OK, BASELINE_VERDICT = check_baseline_against_v12(BASELINE_ROW)
    BASELINE_ROW["note"] = BASELINE_VERDICT
    append_row(BASELINE_ROW)
except Exception as exc:
    BASELINE_VERDICT = f"baseline run FAILED: {exc!r}"
    print(f"\n❌ PART 0 baseline FAILED: {exc!r}")
    traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

PART1_ENABLED = BASELINE_OK or (BASELINE_ROW is not None and not REQUIRE_BASELINE_REPRODUCTION)
if not BASELINE_OK:
    print("\n" + "🚨" * 40)
    print(f"🚨  PART 0 did not pass ({BASELINE_VERDICT}).")
    print("🚨  " + ("Parts 1-2 will NOT run (REQUIRE_BASELINE_REPRODUCTION=True). Fix the reproduction first, or set it "
                   "to False to run the ablation anyway." if REQUIRE_BASELINE_REPRODUCTION else
                   "REQUIRE_BASELINE_REPRODUCTION=False -> Parts 1-2 run anyway; treat their results with suspicion."))
    print("🚨" * 40)
else:
    print(f"\n✅ Part 0 verified (the v12 winner reproduces) — proceeding to Part 1 (loss-weight ablation). "
          f"Elapsed {(time.time() - _sweep_t0) / 60:.1f} min.")




=== PART 0: consolidated Mamba 2 x (d_state 16, d_conv 2) | loss weights (lambda_align, lambda_cons) = (0.1, 0.1) ===
[PART 0] restored / final Mamba parameter count: 874,496 (2 blocks x 437,248) | known winning v12 run C1: 874,496 -> MATCH ✅
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[CONDITION] P0/baseline la=0.1,lc=0.1 | lambda_align=0.1 | lambda_cons=0.1 | fixed: beta=1.0, mlp g(Δτ), 4 heads, Mamba 2 x (d_state 16, d_conv 2)

--- Preprocessing TRAIN split, Ta=64 (frozen strongest-64 bins) ---


Train Class 0 (Ta=64):   0%|          | 0/13 [00:00<?, ?it/s]

[STRONGEST-64 VERIFICATION] audio chunk shape (raw waveform): (5103, 1536) | np.fft.rfft output shape: (5103, 769) | final a_feats (sqrt-compressed, frozen strongest-64 bins) shape: (5103, 64)
[STRONGEST-64 VERIFICATION] Confirmed: features were produced via np.fft.rfft (frequency-domain magnitude spectrum) using the frozen strongest-64 bin indices, NOT the first 64 bins.


Train Class 1 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

Train Class 2 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

...

P0/baseline la=0.1,lc=0.1 Epoch 18/20 | Train Loss: 0.3162 (cls 0.1443 | align 1.2779 | cons 0.4409) | Val Loss: 0.7796 | Val Acc: 0.8648 | Val Macro-F1: 0.8705


P0/baseline la=0.1,lc=0.1 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

P0/baseline la=0.1,lc=0.1 Epoch 19/20 | Train Loss: 0.2930 (cls 0.1225 | align 1.2644 | cons 0.4403) | Val Loss: 0.8428 | Val Acc: 0.8679 | Val Macro-F1: 0.8738


P0/baseline la=0.1,lc=0.1 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

P0/baseline la=0.1,lc=0.1 Epoch 20/20 | Train Loss: 0.3018 (cls 0.1310 | align 1.2678 | cons 0.4403) | Val Loss: 0.7145 | Val Acc: 0.8910 | Val Macro-F1: 0.8951

🎉 P0/baseline la=0.1,lc=0.1 Training Finished in 176.8s | Best Val Macro-F1: 0.9024 (epoch 17)
[LOSS COMPONENTS @ final epoch 20] (lambda_align=0.1, lambda_cons=0.1)
   train: loss_cls 0.1310 | loss_align 1.2678 (weighted 0.1268) | loss_cons 0.4403 (weighted 0.0440) | total 0.3018
   val  : loss_cls 0.4146 | loss_align 2.5460 (weighted 0.2546) | loss_cons 0.4531 (weighted 0.0453) | total 0.7145
[SHAPES best checkpoint] Mamba input (32, 193, 256) -> 2 blocks, each (32, 193, 256) ✅ -> logits (32, 5) ✅
     BI-MODAL AEROSYNC-MAMBA (P0/baseline la=0.1,lc=0.1): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.9080 (90.80%)
  • Macro-F1 Score      : 0.9048
  • Balanced Accuracy   : 0.9022
  • One-vs-Rest AUROC   : 0.9894
  • Learned Audio-IMU Lag (Δ_as) : -0.0291 (normalized units) ≈ -74.6 ms | std over test windows 0.0685
  • L

# Cell 8: PART 1 — ablation: baseline (0.1, 0.1) | no_align (0.0, 0.1) | no_cons (0.1, 0.0); REAL only if |effect| > 2x the ±0.0005 noise floor

In [9]:
# ==============================================================================
# PART 1 — ablation: does each auxiliary loss term actually matter?
#   1. baseline  lambda_align=0.1, lambda_cons=0.1  (= Part 0 run, REUSED, not re-trained)
#   2. no_align  lambda_align=0.0, lambda_cons=0.1  (removes ONLY L_align)
#   3. no_cons   lambda_align=0.1, lambda_cons=0.0  (removes ONLY L_cons)
#
#   L_align ablation effect = baseline macro-F1 - no_align macro-F1
#   L_cons  ablation effect = baseline macro-F1 - no_cons  macro-F1
#
# Verdict rule (strict, consistent with how thin margins were treated before):
# the noise floor is ±NOISE_BAND_F1 = 0.0005 macro-F1, and an effect only counts
# as REAL if it CLEARLY exceeds it — |effect| > 2 x noise floor = 0.0010. Anything
# at or below that is NOT ESTABLISHED. v12 already flagged margins under 2x the
# floor as "thin", and v11's h=1 vs h=4 margin (+0.0008 macro-F1, one test window)
# was not adopted — under this rule it is NOT ESTABLISHED, as it should be.
# ==============================================================================
NOISE_BAND_F1 = 0.0005                       # spread between near-identical same-seed conditions in earlier sweeps
ESTABLISHED_MARGIN_F1 = 2 * NOISE_BAND_F1    # "clearly exceeds the noise floor"
ABLATION_CONDITIONS = [
    ("baseline", 0.1, 0.1),
    ("no_align", 0.0, 0.1),
    ("no_cons",  0.1, 0.0),
]
TERMS = {   # term -> (ablation condition that removes it, index of its weight in (lambda_align, lambda_cons))
    "align": ("no_align", 0),
    "cons":  ("no_cons", 1),
}

# the h=1 vs h=4 reference margin must come out NOT ESTABLISHED under this rule
assert not (0.0008 > ESTABLISHED_MARGIN_F1), "verdict rule would call the +0.0008 h=1-vs-h=4 margin REAL"


def classify_effect(effect):
    # rounded to 1e-6 so float noise in the subtraction can never push a margin of exactly 0.0010 over the line
    return "REAL" if (effect is not None and round(abs(effect), 6) > ESTABLISHED_MARGIN_F1) else "NOT ESTABLISHED"


ABLATION_ROWS = {}
PART1_VERDICTS = {}
ESTABLISHED_WEIGHT = {"align": DEFAULT_LAMBDAS[0], "cons": DEFAULT_LAMBDAS[1]}
ablation_failed = []
PART1_RAN = False

if not PART1_ENABLED:
    print("⛔ PART 1 SKIPPED — Part 0 did not reproduce the v12 winner (see the banner above). "
          "Set REQUIRE_BASELINE_REPRODUCTION=False in the Part 0 cell to run it anyway.")
else:
    PART1_RAN = True
    for i, (cid, la, lc) in enumerate(ABLATION_CONDITIONS, start=1):
        print("\n\n" + "=" * 90)
        print(f"=== PART 1 condition {i}/{len(ABLATION_CONDITIONS)}: {cid} | lambda_align={la} | lambda_cons={lc}"
              + (" (= Part 0 baseline — reused)" if (la, lc) == DEFAULT_LAMBDAS else "")
              + f" | elapsed {(time.time() - _sweep_t0) / 60:.1f} min ===")
        print("=" * 90)
        try:
            ABLATION_ROWS[cid] = run_or_reuse("P1", cid, la, lc,
                                              note="Part 0 baseline" if (la, lc) == DEFAULT_LAMBDAS else "ablation")
        except Exception as exc:
            ablation_failed.append((cid, repr(exc)))
            print(f"\n❌ P1/{cid} FAILED: {exc!r}")
            traceback.print_exc()
            print("↪️  continuing; the effect that depends on this condition cannot be established.")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --- Ablation effects vs the noise floor ---
    base = ABLATION_ROWS.get("baseline")
    print("\n" + "#" * 100)
    print(f"#  PART 1 RESULT — auxiliary loss-term ablation (noise floor ±{NOISE_BAND_F1} macro-F1; REAL requires "
          f"|effect| > {ESTABLISHED_MARGIN_F1:.4f} = 2x floor)")
    print("#" * 100)
    for cid, _, _ in ABLATION_CONDITIONS:
        r = ABLATION_ROWS.get(cid)
        if r is None:
            print(f"#  {cid:<9} FAILED — no result")
            continue
        print(f"#  {cid:<9} (la={r['lambda_align']}, lc={r['lambda_cons']}) macro-F1 {r['macro_f1']:.4f} | acc "
              f"{r['accuracy']:.4f} | bal-acc {r['balanced_accuracy']:.4f} | AUROC {r['macro_auroc']:.4f} | "
              f"final-epoch train cls/align/cons {r['final_train_loss_cls']:.4f} / {r['final_train_loss_align']:.4f}"
              f" / {r['final_train_loss_cons']:.4f} | {r['status']}")
    print("#")
    for term, (cid, idx) in TERMS.items():
        abl = ABLATION_ROWS.get(cid)
        label = f"L_{term} ablation effect"
        if base is None or abl is None:
            effect, verdict = None, "NOT ESTABLISHED"
            print(f"#  {label:<24} = UNDEFINED ({'baseline' if base is None else cid} missing) -> {verdict} "
                  f"(cannot be established without both runs)")
        else:
            effect = base["macro_f1"] - abl["macro_f1"]
            verdict = classify_effect(effect)
            d_win = int(base["n_test_correct"]) - int(abl["n_test_correct"])
            print(f"#  {label:<24} = baseline {base['macro_f1']:.4f} - {cid} {abl['macro_f1']:.4f} = {effect:+.4f} "
                  f"macro-F1 ({d_win:+d} correct test windows | Δ bal-acc "
                  f"{base['balanced_accuracy'] - abl['balanced_accuracy']:+.4f} | Δ AUROC "
                  f"{base['macro_auroc'] - abl['macro_auroc']:+.4f})")
            if verdict == "REAL":
                helps = effect > 0
                print(f"#     -> REAL: |{effect:+.4f}| > {ESTABLISHED_MARGIN_F1:.4f} (clearly exceeds the ±{NOISE_BAND_F1} "
                      f"noise floor). L_{term} " + ("HELPS — removing it lowers macro-F1." if helps else
                                                    "HURTS — removing it RAISES macro-F1."))
            else:
                print(f"#     -> NOT ESTABLISHED: |{effect:+.4f}| ≤ {ESTABLISHED_MARGIN_F1:.4f} — does not clearly exceed "
                      f"the ±{NOISE_BAND_F1} noise floor"
                      + (" (inside the floor itself)" if abs(effect) <= NOISE_BAND_F1 else
                         " (above the floor but under 2x — a thin margin like v11's h=1 vs h=4 +0.0008, not a finding)"))
        # value Part 1 establishes for this term's weight when it is held fixed in Part 2:
        #   REAL & helps -> keep 0.1 | REAL & hurts -> 0.0 | NOT ESTABLISHED -> default 0.1 retained (no change on noise)
        if verdict == "REAL" and effect < 0:
            ESTABLISHED_WEIGHT[term] = 0.0
        else:
            ESTABLISHED_WEIGHT[term] = DEFAULT_LAMBDAS[idx]
        PART1_VERDICTS[term] = {"effect": effect, "verdict": verdict, "established_weight": ESTABLISHED_WEIGHT[term]}
    print("#")
    print(f"#  Weight each term is held at when it is NOT the grid variable in Part 2: "
          f"lambda_align={ESTABLISHED_WEIGHT['align']} | lambda_cons={ESTABLISHED_WEIGHT['cons']}")
    for cid, err in ablation_failed:
        print(f"#  ❌ {cid}: {err}")
    print("#" * 100)




=== PART 1 condition 1/3: baseline | lambda_align=0.1 | lambda_cons=0.1 (= Part 0 baseline — reused) | elapsed 4.8 min ===
[CONFIG] P1/baseline: lambda_align=0.1 | lambda_cons=0.1
♻️  REUSED, NOT RE-RUN: P1/baseline has loss weights identical to already-computed P0/baseline -> copying its metrics (macro_f1=0.9048).
[LOSS COMPONENTS @ final epoch] (reused from P0/baseline) train: loss_cls 0.1310 | loss_align 1.2678 | loss_cons 0.4403
💾 appended P1 / baseline to /kaggle/working/loss_weight_ablation_results.csv (row)


=== PART 1 condition 2/3: no_align | lambda_align=0.0 | lambda_cons=0.1 | elapsed 4.8 min ===
[CONFIG] P1/no_align: lambda_align=0.0 | lambda_cons=0.1
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[CONDITION] P1/no_align la=0.0,lc=0.1 | lambda_align=0.0 | lambda_cons=0.1 | fixed: beta=1.0, mlp g(Δτ), 4 heads, Mamba 2 x (d_state 16, d_conv 2)
[DATA] reusing the Ta=64 windows built for the first condition (identical data for every condition)
DataLoa

P1/no_align la=0.0,lc=0.1 Epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

P1/no_align la=0.0,lc=0.1 Epoch 01/20 | Train Loss: 1.1152 (cls 1.0773 | align 3.3185 | cons 0.3794) | Val Loss: 0.5660 | Val Acc: 0.8208 | Val Macro-F1: 0.8292 ⭐ [Best Saved]


P1/no_align la=0.0,lc=0.1 Epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

P1/no_align la=0.0,lc=0.1 Epoch 02/20 | Train Loss: 0.6787 (cls 0.6127 | align 3.1758 | cons 0.6599) | Val Loss: 0.9615 | Val Acc: 0.6551 | Val Macro-F1: 0.6570


...

P1/no_cons la=0.1,lc=0.0 Epoch 18/20 | Train Loss: 0.2670 (cls 0.1518 | align 1.1529 | cons 1.5439) | Val Loss: 0.7701 | Val Acc: 0.8690 | Val Macro-F1: 0.8743


P1/no_cons la=0.1,lc=0.0 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

P1/no_cons la=0.1,lc=0.0 Epoch 19/20 | Train Loss: 0.2465 (cls 0.1325 | align 1.1400 | cons 1.5509) | Val Loss: 0.8237 | Val Acc: 0.8690 | Val Macro-F1: 0.8741


P1/no_cons la=0.1,lc=0.0 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

P1/no_cons la=0.1,lc=0.0 Epoch 20/20 | Train Loss: 0.2528 (cls 0.1386 | align 1.1421 | cons 1.5558) | Val Loss: 0.7053 | Val Acc: 0.8878 | Val Macro-F1: 0.8921

🎉 P1/no_cons la=0.1,lc=0.0 Training Finished in 153.9s | Best Val Macro-F1: 0.8969 (epoch 17)
[LOSS COMPONENTS @ final epoch 20] (lambda_align=0.1, lambda_cons=0.0)
   train: loss_cls 0.1386 | loss_align 1.1421 (weighted 0.1142) | loss_cons 1.5558 (weighted 0.0000) | total 0.2528
   val  : loss_cls 0.4634 | loss_align 2.4191 (weighted 0.2419) | loss_cons 1.5368 (weighted 0.0000) | total 0.7053
   ↪ loss_cons has weight 0.0: its raw value above is still computed every step but contributes nothing to the total/gradient (epoch-1 train value 0.6111 -> final 1.5558).
[SHAPES best checkpoint] Mamba input (32, 193, 256) -> 2 blocks, each (32, 193, 256) ✅ -> logits (32, 5) ✅
     BI-MODAL AEROSYNC-MAMBA (P1/no_cons la=0.1,lc=0.0): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.8920 (89.20%)
  • Macro-F1 Score      : 0.8883
  • Ba

# Cell 9: PART 2 — conditional grid {0.05, 0.1, 0.2} ONLY for the loss weight(s) Part 1 found REAL (skipped if none)

In [10]:
# ==============================================================================
# PART 2 — conditional grid, ONLY for the term(s) Part 1 found REAL (decided here
# in code from PART1_VERDICTS — zero, one or two terms):
#   grid weight ∈ {0.05, 0.1, 0.2} for that term; 0.1 is REUSED from Part 1 (not
#   re-trained); the other term is held at the value Part 1 established for it
#   (ESTABLISHED_WEIGHT: 0.1, or 0.0 if Part 1 showed removing it genuinely helps).
# If NEITHER term is REAL, no grid is run — there is no established effect to tune.
#
# Best-weight rule (same noise-floor discipline as Part 1): a grid weight replaces
# 0.1 only if it beats the 0.1 point by more than ESTABLISHED_MARGIN_F1; otherwise
# 0.1 is retained and the nominal top is flagged as within noise.
# ==============================================================================
GRID_WEIGHTS = [0.05, 0.1, 0.2]
GRID_CENTER = 0.1
GRID_RESULTS = {}      # term -> {"rows": {weight: row}, "held": (other_term, value), "best_weight", "nominal_top", ...}
grid_failed = []
REAL_TERMS = []
PART2_RAN = False

if not PART1_RAN:
    print("⛔ PART 2 SKIPPED — Part 1 did not run (Part 0 did not reproduce the v12 winner).")
else:
    REAL_TERMS = [t for t in ("align", "cons") if PART1_VERDICTS.get(t, {}).get("verdict") == "REAL"]
    print("PART 2 PLAN (decided from Part 1's verdicts, not configured by hand):")
    for t in ("align", "cons"):
        v = PART1_VERDICTS.get(t, {})
        eff = v.get("effect")
        print(f"  L_{t:<6}: ablation effect {('%+.4f' % eff) if eff is not None else 'undefined'} -> "
              f"{v.get('verdict', 'NOT ESTABLISHED')}" + ("  => GRID" if t in REAL_TERMS else "  => no grid"))

    if not REAL_TERMS:
        print("\n" + "=" * 90)
        print("⏹️  PART 2 NOT RUN — NO GRID NEEDED.")
        print(f"   Neither L_align nor L_cons showed an ablation effect that clearly exceeds the ±{NOISE_BAND_F1} "
              f"macro-F1 noise floor (REAL requires |effect| > {ESTABLISHED_MARGIN_F1:.4f}).")
        print("   Tuning a weight whose effect is not established would only fit noise, so no grid is run.")
        print(f"   The loss weights stay at the default (lambda_align, lambda_cons) = {DEFAULT_LAMBDAS}.")
        print("=" * 90)
    else:
        PART2_RAN = True
        for term in REAL_TERMS:
            other = "cons" if term == "align" else "align"
            held = ESTABLISHED_WEIGHT[other]
            print("\n\n" + "#" * 90)
            print(f"#  PART 2 GRID — lambda_{term} ∈ {GRID_WEIGHTS} | lambda_{other} held at {held} "
                  f"(value Part 1 established: L_{other} {PART1_VERDICTS[other]['verdict']})")
            print("#" * 90)
            rows = {}
            for w in GRID_WEIGHTS:
                la, lc = (w, held) if term == "align" else (held, w)
                is_center = (w == GRID_CENTER)
                print("\n" + "=" * 90)
                print(f"=== P2/{term} lambda_{term}={w} | lambda_align={la} | lambda_cons={lc}"
                      + (" (grid centre — reused from Part 1)" if is_center else "")
                      + f" | elapsed {(time.time() - _sweep_t0) / 60:.1f} min ===")
                print("=" * 90)
                if is_center and (float(la), float(lc)) not in _RESULTS_BY_LAMBDAS:
                    print(f"⚠️  grid centre ({la}, {lc}) has no Part 1 result to reuse (its Part 1 run failed) — "
                          f"training it now.")
                try:
                    rows[w] = run_or_reuse("P2", f"{term}_{w}", la, lc,
                                           note=f"grid lambda_{term}; lambda_{other} held at {held}")
                except Exception as exc:
                    grid_failed.append((f"{term}_{w}", repr(exc)))
                    print(f"\n❌ P2/{term}_{w} FAILED: {exc!r}")
                    traceback.print_exc()
                    print("↪️  continuing; this grid point is excluded.")
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            # --- this term's grid result ---
            print("\n" + "#" * 100)
            print(f"#  PART 2 RESULT — lambda_{term} grid (lambda_{other}={held}; noise floor ±{NOISE_BAND_F1}, "
                  f"a weight beats {GRID_CENTER} only by > {ESTABLISHED_MARGIN_F1:.4f})")
            print("#" * 100)
            centre = rows.get(GRID_CENTER)
            for w in GRID_WEIGHTS:
                r = rows.get(w)
                if r is None:
                    print(f"#  lambda_{term}={w:<5} FAILED")
                    continue
                d = "" if (centre is None or w == GRID_CENTER) else f" | Δ vs {GRID_CENTER}: {r['macro_f1'] - centre['macro_f1']:+.4f}"
                print(f"#  lambda_{term}={w:<5} macro-F1 {r['macro_f1']:.4f} | acc {r['accuracy']:.4f} | bal-acc "
                      f"{r['balanced_accuracy']:.4f} | AUROC {r['macro_auroc']:.4f} | final-epoch train "
                      f"cls/align/cons {r['final_train_loss_cls']:.4f} / {r['final_train_loss_align']:.4f} / "
                      f"{r['final_train_loss_cons']:.4f} | {r['status']}{d}")
            ok_rows = {w: r for w, r in rows.items() if r is not None}
            best_weight, nominal_top, flag = None, None, None
            if ok_rows:
                nominal_top = max(ok_rows, key=lambda w: (ok_rows[w]["macro_f1"], w == GRID_CENTER))  # ties -> centre
                if centre is None:
                    best_weight = nominal_top
                    flag = f"centre {GRID_CENTER} missing — nominal top taken without a noise-floor comparison"
                elif nominal_top == GRID_CENTER:
                    best_weight = GRID_CENTER
                else:
                    margin = ok_rows[nominal_top]["macro_f1"] - centre["macro_f1"]
                    if classify_effect(margin) == "REAL":
                        best_weight = nominal_top
                    else:
                        best_weight = GRID_CENTER
                        flag = (f"nominal top lambda_{term}={nominal_top} beats {GRID_CENTER} by only {margin:+.4f} "
                                f"(≤ {ESTABLISHED_MARGIN_F1:.4f}) -> NOT ESTABLISHED; {GRID_CENTER} RETAINED")
                print("#")
                print(f"#  🏁 BEST lambda_{term} = {best_weight}  (macro-F1 {ok_rows[best_weight]['macro_f1']:.4f}; "
                      f"nominal top {nominal_top})")
                if flag:
                    print(f"#  ⚠️  THIN / NOISE FLAG: {flag}")
            print("#" * 100)
            GRID_RESULTS[term] = {"rows": rows, "held": (other, held), "best_weight": best_weight,
                                  "nominal_top": nominal_top, "flag": flag}

        print("\n\n" + "=" * 90)
        print(f"PART 2 COMPLETE in {(time.time() - _sweep_t0) / 60:.1f} min (incl. Parts 0-1) | results: {RESULTS_CSV}")
        print(f"Terms gridded: {', '.join('L_' + t for t in REAL_TERMS)}")
        for fid, err in grid_failed:
            print(f"  ❌ {fid}: {err}")
        print("=" * 90)


PART 2 PLAN (decided from Part 1's verdicts, not configured by hand):
  L_align : ablation effect +0.0088 -> REAL  => GRID
  L_cons  : ablation effect +0.0165 -> REAL  => GRID


##########################################################################################
#  PART 2 GRID — lambda_align ∈ [0.05, 0.1, 0.2] | lambda_cons held at 0.1 (value Part 1 established: L_cons REAL)
##########################################################################################

=== P2/align lambda_align=0.05 | lambda_align=0.05 | lambda_cons=0.1 | elapsed 13.2 min ===
[CONFIG] P2/align_0.05: lambda_align=0.05 | lambda_cons=0.1
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[CONDITION] P2/align_0.05 la=0.05,lc=0.1 | lambda_align=0.05 | lambda_cons=0.1 | fixed: beta=1.0, mlp g(Δτ), 4 heads, Mamba 2 x (d_state 16, d_conv 2)
[DATA] reusing the Ta=64 windows built for the first condition (identical data for every condition)
DataLoaders ready -> Train: 4934, Val: 954, Test: 

P2/align_0.05 la=0.05,lc=0.1 Epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

P2/align_0.05 la=0.05,lc=0.1 Epoch 01/20 | Train Loss: 1.2703 (cls 1.0740 | align 3.1876 | cons 0.3693) | Val Loss: 0.7292 | Val Acc: 0.8176 | Val Macro-F1: 0.8257 ⭐ [Best Saved]


P2/align_0.05 la=0.05,lc=0.1 Epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

P2/align_0.05 la=0.05,lc=0.1 Epoch 02/20 | Train Loss: 0.8055 (cls 0.6016 | align 2.8171 | cons 0.6297) | Val Loss: 1.1290 | Val Acc: 0.6698 | Val Macro-F1: 0.6747


...

P2/cons_0.2 la=0.1,lc=0.2 Epoch 18/20 | Train Loss: 0.3484 (cls 0.1382 | align 1.4172 | cons 0.3425) | Val Loss: 0.8525 | Val Acc: 0.8606 | Val Macro-F1: 0.8671


P2/cons_0.2 la=0.1,lc=0.2 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

P2/cons_0.2 la=0.1,lc=0.2 Epoch 19/20 | Train Loss: 0.3331 (cls 0.1249 | align 1.3989 | cons 0.3413) | Val Loss: 0.9068 | Val Acc: 0.8606 | Val Macro-F1: 0.8667


P2/cons_0.2 la=0.1,lc=0.2 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

P2/cons_0.2 la=0.1,lc=0.2 Epoch 20/20 | Train Loss: 0.3355 (cls 0.1272 | align 1.4022 | cons 0.3403) | Val Loss: 0.7690 | Val Acc: 0.8795 | Val Macro-F1: 0.8849

🎉 P2/cons_0.2 la=0.1,lc=0.2 Training Finished in 162.7s | Best Val Macro-F1: 0.9066 (epoch 17)
[LOSS COMPONENTS @ final epoch 20] (lambda_align=0.1, lambda_cons=0.2)
   train: loss_cls 0.1272 | loss_align 1.4022 (weighted 0.1402) | loss_cons 0.3403 (weighted 0.0681) | total 0.3355
   val  : loss_cls 0.4301 | loss_align 2.6777 (weighted 0.2678) | loss_cons 0.3554 (weighted 0.0711) | total 0.7690
[SHAPES best checkpoint] Mamba input (32, 193, 256) -> 2 blocks, each (32, 193, 256) ✅ -> logits (32, 5) ✅
     BI-MODAL AEROSYNC-MAMBA (P2/cons_0.2 la=0.1,lc=0.2): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.9088 (90.88%)
  • Macro-F1 Score      : 0.9056
  • Balanced Accuracy   : 0.9029
  • One-vs-Rest AUROC   : 0.9896
  • Learned Audio-IMU Lag (Δ_as) : -0.0333 (normalized units) ≈ -85.3 ms | std over test windows 0.0749
  • L

# Cell 10: Final Summary — Part 0 verdict, full table, Part 1 verdicts, Part 2 best weight(s), overall best configuration vs the Part 0 baseline

In [11]:
# ==============================================================================
# FINAL SUMMARY — full table reloaded from the CSV on disk (crash-safe source of
# truth) + Part 0 verdict, Part 1 verdicts, Part 2 best weights, and the overall
# best configuration across Parts 1-2, compared against the Part 0 baseline.
# ==============================================================================
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"{RESULTS_CSV} does not exist — no condition finished. Re-run the Part 0-2 cells.")

results_df = pd.read_csv(RESULTS_CSV)
pd.set_option("display.width", 260)
try:
    from IPython.display import display
except Exception:
    display = lambda t: print(t.to_string(index=False))

show_cols = ["stage", "condition_id", "lambda_align", "lambda_cons", "accuracy", "macro_f1", "balanced_accuracy",
             "macro_auroc", "final_train_loss_cls", "final_train_loss_align", "final_train_loss_cons",
             "best_epoch", "train_seconds", "status"]
fmt = results_df[show_cols].copy()
for col in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
            "final_train_loss_cls", "final_train_loss_align", "final_train_loss_cons"):
    fmt[col] = fmt[col].round(4)

# --- 1. Part 0 ---
print("=" * 120)
p0 = results_df[results_df["stage"] == "P0"]
if len(p0):
    r = p0.iloc[0]
    print(f"PART 0 — consolidated Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) "
          f"[{int(r['mamba_params']):,} Mamba params] vs the v12 winner:")
    for k, v in V12_BASELINE.items():
        d = r[k] - v
        print(f"  {k:<20} v12 {v:.4f} | this run {r[k]:.4f} | delta {d:+.4f}" + ("" if abs(d) <= REPRO_TOL else "   <-- differs"))
    print(f"  verdict: {r['note']}")
    if not str(r["note"]).startswith("reproduces"):
        print("🚨" * 40 + "\n🚨  PART 0 did NOT reproduce the v12 winner — Part 1/2 results (if any) are not anchored.\n"
              + "🚨" * 40)
else:
    print("🚨 PART 0 row missing from the CSV — the baseline did not finish.")

# --- 2. Full table (every row, including reused ones) ---
print("\n" + "=" * 120)
print("   FULL RESULTS — every condition in run order (♻️ reused rows copy an identical earlier condition; "
      "loss components are RAW final-epoch train averages)")
print("=" * 120)
display(fmt)

# every row: metrics either from a live forward pass or explicitly marked reused
for _, row in results_df.iterrows():
    assert_metrics_provenance(row.to_dict())
print(f"✅ metrics provenance: {int((results_df['status'] == 'computed').sum())} row(s) from a live test-set forward "
      f"pass, {int((results_df['status'] != 'computed').sum())} row(s) explicitly marked as reused")

# --- 3. Part 1 ---
print("\n" + "#" * 120)
if PART1_RAN:
    print(f"#  PART 1 — ablation verdicts (REAL requires |effect| > {ESTABLISHED_MARGIN_F1:.4f} macro-F1 = 2x the "
          f"±{NOISE_BAND_F1} noise floor)")
    for t in ("align", "cons"):
        v = PART1_VERDICTS.get(t, {})
        eff = v.get("effect")
        print(f"#    L_{t:<6} ablation effect {('%+.4f' % eff) if eff is not None else 'undefined':>8} -> "
              f"{v.get('verdict', 'NOT ESTABLISHED')}"
              + ((" (helps)" if eff > 0 else " (hurts)") if v.get("verdict") == "REAL" else ""))
    found = [f"L_{t}" for t in REAL_TERMS]
    print(f"#  Term(s) found to matter: {', '.join(found) if found else 'NONE'}")
else:
    print("#  PART 1 did not run (see the Part 0 verdict above).")

# --- 4. Part 2 ---
print("#")
if PART2_RAN:
    for t, g in GRID_RESULTS.items():
        bw = g["best_weight"]
        best_row = g["rows"].get(bw) if bw is not None else None
        print(f"#  PART 2 — lambda_{t} grid {GRID_WEIGHTS} (lambda_{g['held'][0]} held at {g['held'][1]}): best "
              f"weight {bw}" + (f" (macro-F1 {best_row['macro_f1']:.4f})" if best_row is not None else "")
              + (f" | ⚠️ {g['flag']}" if g["flag"] else ""))
elif PART1_RAN:
    print("#  PART 2 — not run: neither ablation cleared the noise floor, so there was no established effect to grid.")
else:
    print("#  PART 2 — not run (Part 1 did not run).")

# --- 5. Overall best configuration across Parts 1-2 vs the Part 0 baseline ---
print("#")
computed = results_df[results_df["status"] == "computed"]
if len(p0) and len(computed):
    base_r = p0.iloc[0]
    ranked = computed.assign(_is_base=(computed["stage"] == "P0")).sort_values(
        ["macro_f1", "_is_base"], ascending=[False, False], kind="stable")
    top = ranked.iloc[0]
    margin = float(top["macro_f1"] - base_r["macro_f1"])
    print(f"#  Unique trained configurations by macro-F1:")
    for _, r in ranked.iterrows():
        print(f"#    (lambda_align={r['lambda_align']}, lambda_cons={r['lambda_cons']}) macro-F1 {r['macro_f1']:.4f} | "
              f"acc {r['accuracy']:.4f} | bal-acc {r['balanced_accuracy']:.4f} | AUROC {r['macro_auroc']:.4f} | "
              f"Δ macro-F1 vs Part 0 {r['macro_f1'] - base_r['macro_f1']:+.4f}  [{r['stage']}/{r['condition_id']}]")
    print("#")
    if top["stage"] == "P0" or classify_effect(margin) != "REAL":
        print(f"#  🏆 OVERALL BEST: the Part 0 baseline (lambda_align, lambda_cons) = {DEFAULT_LAMBDAS} is RETAINED — "
              f"macro-F1 {base_r['macro_f1']:.4f}.")
        if top["stage"] != "P0":
            print(f"#     nominal top (lambda_align={top['lambda_align']}, lambda_cons={top['lambda_cons']}) "
                  f"macro-F1 {top['macro_f1']:.4f} is only {margin:+.4f} vs Part 0 (≤ {ESTABLISHED_MARGIN_F1:.4f}) "
                  f"-> NOT ESTABLISHED as better.")
    else:
        print(f"#  🏆 OVERALL BEST: (lambda_align={top['lambda_align']}, lambda_cons={top['lambda_cons']}) — macro-F1 "
              f"{top['macro_f1']:.4f} vs Part 0 baseline {base_r['macro_f1']:.4f} ({margin:+.4f}, REAL: > "
              f"{ESTABLISHED_MARGIN_F1:.4f}) | acc {top['accuracy']:.4f} ({top['accuracy'] - base_r['accuracy']:+.4f}) | "
              f"bal-acc {top['balanced_accuracy']:.4f} ({top['balanced_accuracy'] - base_r['balanced_accuracy']:+.4f}) | "
              f"AUROC {top['macro_auroc']:.4f} ({top['macro_auroc'] - base_r['macro_auroc']:+.4f})")
else:
    print("#  No overall comparison possible — the Part 0 baseline did not finish.")
print("#" * 120)

print("\n   Caveat: same-seed reruns here are bit-exact, so the noise floor is the spread between near-identical "
      "conditions, not a measured seed-to-seed variance; one seed per condition cannot establish a small effect.")
print(f"📄 full results CSV: {RESULTS_CSV}")


PART 0 — consolidated Mamba 2 x (d_state 16, d_conv 2) [874,496 Mamba params] vs the v12 winner:
  accuracy             v12 0.9080 | this run 0.9080 | delta -0.0000
  macro_f1             v12 0.9048 | this run 0.9048 | delta -0.0000
  balanced_accuracy    v12 0.9022 | this run 0.9022 | delta -0.0000
  macro_auroc          v12 0.9894 | this run 0.9894 | delta -0.0000
  verdict: reproduces v12 winner (all 4 metrics within 0.0005)

   FULL RESULTS — every condition in run order (♻️ reused rows copy an identical earlier condition; loss components are RAW final-epoch train averages)


,stage,condition_id,lambda_align,lambda_cons,accuracy,macro_f1,balanced_accuracy,macro_auroc,final_train_loss_cls,final_train_loss_align,final_train_loss_cons,best_epoch,train_seconds,status
0,P0,baseline,0.10,0.10,0.9080,0.9048,0.9022,0.9894,0.1310,1.2678,0.4403,17,176.8,computed
1,P1,baseline,0.10,0.10,0.9080,0.9048,0.9022,0.9894,0.1310,1.2678,0.4403,17,0.0,reused from P0/baseline (not re-run)
2,P1,no_align,0.00,0.10,0.9000,0.8959,0.8939,0.9878,0.1500,2.9337,0.4327,17,182.6,computed
3,P1,no_cons,0.10,0.00,0.8920,0.8883,0.8859,0.9876,0.1386,1.1421,1.5558,17,153.9,computed
4,P2,align_0.05,0.05,0.10,0.8823,0.8777,0.8755,0.9847,0.1301,2.1091,0.4399,17,144.9,computed
5,P2,align_0.1,0.10,0.10,0.9080,0.9048,0.9022,0.9894,0.1310,1.2678,0.4403,17,0.0,reused from P0/baseline (not re-run)
6,P2,align_0.2,0.20,0.10,0.9009,0.8978,0.8941,0.9876,0.1097,0.5556,0.5268,17,154.2,computed
7,P2,cons_0.05,0.10,0.05,0.8850,0.8815,0.8774,0.9879,0.1367,1.2098,0.5156,13,143.1,computed
8,P2,cons_0.1,0.10,0.10,0.9080,0.9048,0.9022,0.9894,0.1310,1.2678,0.4403,17,0.0,reused from P0/baseline (not re-run)
9,P2,cons_0.2,0.10,0.20,0.9088,0.9056,0.9029,0.9896,0.1272,1.4022,0.3403,17,162.7,computed


✅ metrics provenance: 7 row(s) from a live test-set forward pass, 3 row(s) explicitly marked as reused

########################################################################################################################
#  PART 1 — ablation verdicts (REAL requires |effect| > 0.0010 macro-F1 = 2x the ±0.0005 noise floor)
#    L_align  ablation effect  +0.0088 -> REAL (helps)
#    L_cons   ablation effect  +0.0165 -> REAL (helps)
#  Term(s) found to matter: L_align, L_cons
#
#  PART 2 — lambda_align grid [0.05, 0.1, 0.2] (lambda_cons held at 0.1): best weight 0.1 (macro-F1 0.9048)
#  PART 2 — lambda_cons grid [0.05, 0.1, 0.2] (lambda_align held at 0.1): best weight 0.1 (macro-F1 0.9048) | ⚠️ nominal top lambda_cons=0.2 beats 0.1 by only +0.0008 (≤ 0.0010) -> NOT ESTABLISHED; 0.1 RETAINED
#
#  Unique trained configurations by macro-F1:
#    (lambda_align=0.1, lambda_cons=0.2) macro-F1 0.9056 | acc 0.9088 | bal-acc 0.9029 | AUROC 0.9896 | Δ macro-F1 vs Part 0 +0.0008  [P2/cons_0.2]
# 